### https://www.kaggle.com/datasets/msambare/fer2013
Membre du groupe : 
Thomas Lefebvre ; 
Junior ; 
Jack ; 
Kemuel ; 
Mory ; 
Adolphe

In [1]:
import kagglehub
from pathlib import Path

# Récupérer le dossier où se trouve le script Python
BASE_DIR = Path.cwd()


# Créer le chemin vers le dossier data
DATA_DIR = BASE_DIR / "data"

# Télécharger le dataset dans ce dossier
path = kagglehub.dataset_download(
    "msambare/fer2013",
    output_dir=str(DATA_DIR)
)

print("Dataset téléchargé dans :", path)

## Etape importante afin que le code fonctionne sur Colab


/Users/thom_lfbv/Documents/01_ETUDES/B3_ECE/ML/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset téléchargé dans : /Users/thom_lfbv/Documents/01_ETUDES/B3_ECE/ML/data


## Chargement du data set

In [4]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

# Prétraitement des images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor()
])

# Chargement des datasets
dataset = datasets.ImageFolder("./data/train", transform=transform)
test_ds = datasets.ImageFolder("./data/test", transform=transform)

# Répartition du train : 85 % entraînement, 15 % validation
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Vérification
print(f"Train : {len(train_ds)} images")
print(f"Validation : {len(val_ds)} images")
print(f"Test : {len(test_ds)} images")

Train : 24402 images
Validation : 4307 images
Test : 7178 images


Nous avons dans le dataset , un dossier train et un dossier test.
Le data set validation permettra une première évaluation de notre modèle avec l'ajustement des hyperparamètres.
Le data set test permettra une évaluation finale de notre modèle.


## Conversion en vecteur

In [5]:
from torch.utils.data import DataLoader



# Création des DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("DataLoaders créés avec succès !")

# Vérification
print("Train :", len(train_loader))
print("Validation :", len(val_loader))
print("Test :", len(test_loader))

DataLoaders créés avec succès !
Train : 382
Validation : 68
Test : 113


Ces vérifications sont cohérentes nous avons créer des lots de 64 images.


On a mis shuffle = True pour le train afin que à chaque epoch les images soient dans un ordre différent.


On a mis shuffle = False pour le validation et le test afin de pouvoir comparer les résultats de manière plus stable.


## Regressions Logistiques

Avec pytorch on utilise un modèle linéaire que l'on adapte directement.

In [7]:
import torch
import torch.nn as nn

class LogisticRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(48 * 48, 7)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        return self.linear(x)

In [11]:
# Entrainement : 

model = LogisticRegression()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.LBFGS(model.parameters(), lr=0.1)

# Une version plus simple pour commencer :
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/20 - Loss: 1.7944
Epoch 2/20 - Loss: 1.7267
Epoch 3/20 - Loss: 1.7047
Epoch 4/20 - Loss: 1.6912
Epoch 5/20 - Loss: 1.6825
Epoch 6/20 - Loss: 1.6818
Epoch 7/20 - Loss: 1.6639
Epoch 8/20 - Loss: 1.6573
Epoch 9/20 - Loss: 1.6487
Epoch 10/20 - Loss: 1.6630
Epoch 11/20 - Loss: 1.6438
Epoch 12/20 - Loss: 1.6397
Epoch 13/20 - Loss: 1.6338
Epoch 14/20 - Loss: 1.6287
Epoch 15/20 - Loss: 1.6319
Epoch 16/20 - Loss: 1.6348
Epoch 17/20 - Loss: 1.6212
Epoch 18/20 - Loss: 1.6315
Epoch 19/20 - Loss: 1.6174
Epoch 20/20 - Loss: 1.6154


In [14]:
# Evaluation : 
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Validation accuracy : {accuracy:.2%}")

Validation accuracy : 32.90%


Premier résultat avec pytorch : sans aucune optimisation , accuracy = 32.9%
